<center>
<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-RP0101EN-Coursera/v2/M5_Final/images/SN_web_lightmode.png" width="300">
</center>


<h1>Lab for Final Project - Data Analytics for Canadian Crop Production Data Set</h1>

## Introduction

This project analyzes crop production and prices in Canada, along with exchange rates, using R and SQLite.


In [1]:
# Load RSQLite library
library(RSQLite)


In [2]:
# Connect to SQLite database
conn <- dbConnect(SQLite(), dbname="FinalDB.sqlite")


## Problem 1. Create Tables
Create the following tables in your instance:
1. **CROP_DATA**
2. **FARM_PRICES**
3. **DAILY_FX**
4. **MONTHLY_FX**


In [3]:
# Create CROP_DATA table
dbExecute(conn, "DROP TABLE IF EXISTS CROP_DATA")
dbExecute(conn, "CREATE TABLE CROP_DATA (
    CD_ID INTEGER NOT NULL,
    YEAR DATE NOT NULL,
    CROP_TYPE VARCHAR(20) NOT NULL,
    GEO VARCHAR(20) NOT NULL, 
    SEEDED_AREA INTEGER NOT NULL,
    HARVESTED_AREA INTEGER NOT NULL,
    PRODUCTION INTEGER NOT NULL,
    AVG_YIELD INTEGER NOT NULL,
    PRIMARY KEY (CD_ID)
)")

# Create FARM_PRICES table
dbExecute(conn, "DROP TABLE IF EXISTS FARM_PRICES")
dbExecute(conn, "CREATE TABLE FARM_PRICES (
    CD_ID INTEGER NOT NULL,
    DATE DATE NOT NULL,
    CROP_TYPE VARCHAR(20) NOT NULL,
    GEO VARCHAR(20) NOT NULL, 
    PRICE_PRERMT INTEGER NOT NULL,
    PRIMARY KEY (CD_ID)
)")

# Create DAILY_FX table
dbExecute(conn, "DROP TABLE IF EXISTS DAILY_FX")
dbExecute(conn, "CREATE TABLE DAILY_FX (
    DFX_ID INTEGER NOT NULL,
    DATE DATE NOT NULL, 
    FXUSDCAD FLOAT(6),
    PRIMARY KEY (DFX_ID)
)")

# Create MONTHLY_FX table
dbExecute(conn, "DROP TABLE IF EXISTS MONTHLY_FX")
dbExecute(conn, "CREATE TABLE MONTHLY_FX (
    DFX_ID INTEGER NOT NULL,
    DATE DATE NOT NULL, 
    FXUSDCAD FLOAT(6),
    PRIMARY KEY (DFX_ID)
)")

print("Tables created successfully.")


Tables created successfully.


## Problem 2. Read Datasets and Load Tables
Read the datasets and load them into the tables.


In [4]:
# Read datasets from URLs
crop_df <- read.csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-RP0203EN-SkillsNetwork/labs/Final%20Project/Annual_Crop_Data.csv')
farm_prices <- read.csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-RP0203EN-SkillsNetwork/labs/Final%20Project/Monthly_Farm_Prices.csv')
fx_df <- read.csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-RP0203EN-SkillsNetwork/labs/Final%20Project/Daily_FX.csv')
monthly_fx <- read.csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-RP0203EN-SkillsNetwork/labs/Final%20Project/Monthly_FX.csv')

# Load them into SQLite tables
dbWriteTable(conn, "CROP_DATA", crop_df, append=TRUE, row.names=FALSE)
dbWriteTable(conn, "FARM_PRICES", farm_prices, append=TRUE, row.names=FALSE)
dbWriteTable(conn, "DAILY_FX", fx_df, append=TRUE, row.names=FALSE)
dbWriteTable(conn, "MONTHLY_FX", monthly_fx, append=TRUE, row.names=FALSE)

print("Datasets loaded successfully.")


Datasets loaded successfully.


## Problem 3. How many records are in the farm prices dataset?


In [5]:
dbGetQuery(conn, "SELECT COUNT(*) AS RECORD_COUNT FROM FARM_PRICES")


,RECORD_COUNT
1,2678


## Problem 4. Which geographies are included in the farm prices dataset?


In [6]:
dbGetQuery(conn, "SELECT DISTINCT GEO FROM FARM_PRICES")


,GEO
1,Alberta
2,Saskatchewan


## Problem 5. How many hectares of Rye were harvested in Canada in 1968?


In [7]:
dbGetQuery(conn, "SELECT SUM(HARVESTED_AREA) AS TOTAL_RYE_HARVESTED FROM CROP_DATA WHERE CROP_TYPE='Rye' AND GEO='Canada' AND YEAR LIKE '1968%'")


,TOTAL_RYE_HARVESTED
1,274100


## Problem 6. Query and display the first 6 rows of the farm prices table for Rye.


In [8]:
dbGetQuery(conn, "SELECT * FROM FARM_PRICES WHERE CROP_TYPE='Rye' LIMIT 6")


,CD_ID,DATE,CROP_TYPE,GEO,PRICE_PRERMT
1,4,1985-01-01,Rye,Alberta,100.77
2,5,1985-01-01,Rye,Saskatchewan,109.75
3,10,1985-02-01,Rye,Alberta,95.05
4,11,1985-02-01,Rye,Saskatchewan,103.46
5,16,1985-03-01,Rye,Alberta,96.77
6,17,1985-03-01,Rye,Saskatchewan,106.38


## Problem 7. Which provinces grew Barley?


In [9]:
dbGetQuery(conn, "SELECT DISTINCT GEO FROM CROP_DATA WHERE CROP_TYPE='Barley' AND GEO != 'Canada'")


,GEO
1,Alberta
2,Saskatchewan


## Problem 8. Find the first and last dates for the farm prices data.


In [10]:
dbGetQuery(conn, "SELECT MIN(DATE) AS FIRST_DATE, MAX(DATE) AS LAST_DATE FROM FARM_PRICES")


,FIRST_DATE,LAST_DATE
1,1985-01-01,2020-12-01


## Problem 9. Which crops have ever reached a farm price greater than or equal to $350 per metric tonne?


In [11]:
dbGetQuery(conn, "SELECT DISTINCT CROP_TYPE FROM FARM_PRICES WHERE PRICE_PRERMT >= 350")


,CROP_TYPE
1,Canola


## Problem 10. Rank the crop types harvested in Saskatchewan in the year 2000 by their average yield. Which crop performed best?


In [12]:
dbGetQuery(conn, "SELECT CROP_TYPE, AVG_YIELD FROM CROP_DATA WHERE GEO='Saskatchewan' AND YEAR LIKE '2000%' ORDER BY AVG_YIELD DESC")


,CROP_TYPE,AVG_YIELD
1,Barley,2800
2,Wheat,2200
3,Rye,2100
4,Canola,1400


## Problem 11. Rank the crops and geographies by their average yield (KG per hectare) since the year 2000. Which crop and province had the highest average yield since the year 2000?


In [13]:
dbGetQuery(conn, "SELECT CROP_TYPE, GEO, AVG(AVG_YIELD) AS AVERAGE_YIELD FROM CROP_DATA WHERE GEO != 'Canada' AND YEAR >= '2000' GROUP BY CROP_TYPE, GEO ORDER BY AVERAGE_YIELD DESC")


,CROP_TYPE,GEO,AVERAGE_YIELD
1,Barley,Alberta,2922.0535714285716
2,Barley,Saskatchewan,2559.0535714285716
3,Wheat,Alberta,2465.410714285714
4,Rye,Alberta,2141.6071428571427
5,Wheat,Saskatchewan,2026.642857142857
6,Rye,Saskatchewan,1765.1964285714287
7,Canola,Alberta,1478.017857142857
8,Canola,Saskatchewan,1388.6964285714287


## Problem 12. Use a subquery to determine how much wheat was harvested in Canada in the most recent year of the data.


In [14]:
dbGetQuery(conn, "SELECT SUM(HARVESTED_AREA) AS TOTAL_WHEAT_HARVESTED FROM CROP_DATA WHERE CROP_TYPE='Wheat' AND GEO='Canada' AND YEAR = (SELECT MAX(YEAR) FROM CROP_DATA)")


,TOTAL_WHEAT_HARVESTED
1,10017800


## Problem 13. Use an implicit inner join to calculate the monthly price per metric tonne of Canola grown in Saskatchewan in both Canadian and US dollars. Display the most recent 6 months of the data.


In [15]:
dbGetQuery(conn, "SELECT FARM_PRICES.DATE, FARM_PRICES.PRICE_PRERMT AS PRICE_CAD, (FARM_PRICES.PRICE_PRERMT / MONTHLY_FX.FXUSDCAD) AS PRICE_USD FROM FARM_PRICES INNER JOIN MONTHLY_FX ON FARM_PRICES.DATE = MONTHLY_FX.DATE WHERE FARM_PRICES.CROP_TYPE = 'Canola' AND FARM_PRICES.GEO = 'Saskatchewan' ORDER BY FARM_PRICES.DATE DESC LIMIT 6")


,DATE,PRICE_CAD,PRICE_USD
1,2020-12-01,507.33,396.1128336506638
2,2020-11-01,495.64,379.2718201435545
3,2020-10-01,474.8,359.2964551335752
4,2020-09-01,463.52,350.4057020986492
5,2020-08-01,464.6,351.3827280943575
6,2020-07-01,462.88,342.9121754268993
